<a href="https://colab.research.google.com/github/yoshiagent/vibration-colab/blob/main/notebooks/Sinusoids_Beating.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="在 Colab 開啟"/></a>

# 弦波、相量與拍 Sinusoids, Phasors and Beating

對照網頁：[振動學｜弦波與相量](https://yoshiagent.github.io/vibration/SIN.html)

| 原檔 | 內容 | 本筆記本 |
|---|---|---|
| `phasor.m` | 旋轉相量在虛軸上的投影＝弦波（MATLAB 逐格動畫） | 第 1 節（Plotly 動畫，按 ▶ 播放） |
| `phasedif.m` | 兩個同頻率弦波的相位差 | 第 2 節 |
| （網頁 phasor3） | 同頻率弦波相加＝相量相加 | 第 3 節 |
| `beatsound.m` | 兩弦波相乘（拍）的波形與聲音 | 第 4 節 |

只用 Colab 內建的 numpy、plotly（聲音用 IPython 的 `Audio`）。圖上的文字用英文。

In [1]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import Audio, display


def show(fig, title, xlab="t (s)", ylab="", h=420):
    fig.update_layout(title=title, height=h, template="plotly_white", margin=dict(l=60, r=20, t=50, b=50))
    if xlab:
        fig.update_xaxes(title_text=xlab)
    if ylab:
        fig.update_yaxes(title_text=ylab)
    fig.show()

## 1. 相量與弦波（`phasor.m`）

原程式：
```matlab
t=0:dt:stopt;  x=cos(t);  y=sin(t);           % dt=0.1, stopt=2.3*pi
for ii=1:length(t)
   subplot(1,2,1)  % 單位圓上的旋轉向量 [0 x1],[0 y1]，與逐漸變大的螺線 xtheta,ytheta
   subplot(1,2,2)  % 向量的虛部 sin(theta) 對時間畫出
   drawnow
end
```
長度 1 的向量以角速度 1 rad/s 旋轉，角度 $\theta=t$；向量的實部是 $\cos\theta$、虛部是 $\sin\theta$。
右圖把虛部對時間畫出，就是弦波 $\sin t$。原程式用空迴圈延遲做動畫，這裡改成 Plotly 動畫（按 ▶）。

In [2]:
dt, stopt = 0.1, 2.3 * np.pi
t = np.arange(0, stopt + 1e-9, dt)
x, y = np.cos(t), np.sin(t)
r = 0.1 + 0.00002 * np.cumsum(np.arange(1, len(t) + 1))     # 原程式的螺線半徑 r=r+0.00002*ii（只是裝飾）

fig = make_subplots(rows=1, cols=2, subplot_titles=("phasor e^{j theta}", "imaginary part = sin(theta)"))
fig.add_trace(go.Scatter(x=x, y=y, mode="lines", line=dict(color="#ddd", dash="dot"), name="unit circle"), 1, 1)
fig.add_trace(go.Scatter(x=t, y=y, mode="lines", line=dict(color="#ddd", dash="dot"), name="sin t"), 1, 2)
frames = []
for i in range(len(t)):
    frames.append(go.Frame(name=str(i), data=[
        go.Scatter(x=[0, x[i]], y=[0, y[i]], mode="lines+markers", line=dict(color="red", width=3), name="phasor"),
        go.Scatter(x=[x[i], 1.25], y=[y[i], y[i]], mode="lines", line=dict(color="red", dash="dot"), name="projection"),
        go.Scatter(x=t[:i + 1], y=y[:i + 1], mode="lines", line=dict(color="blue"), name="sin t traced"),
        go.Scatter(x=[t[i], t[i]], y=[0, y[i]], mode="lines+markers", line=dict(color="green", dash="dot"), name="sin(theta)"),
    ], traces=[2, 3, 4, 5]))
for d, (row, col) in zip(frames[0].data, [(1, 1), (1, 1), (1, 2), (1, 2)]):
    fig.add_trace(d, row, col)
fig.frames = frames
fig.update_layout(
    updatemenus=[dict(type="buttons", x=0, y=-0.15, buttons=[
        dict(label="▶ Play", method="animate", args=[None, dict(frame=dict(duration=60, redraw=True), fromcurrent=True)]),
        dict(label="❚❚ Pause", method="animate", args=[[None], dict(mode="immediate", frame=dict(duration=0))])])],
    sliders=[dict(x=0.15, len=0.85, y=-0.1, currentvalue=dict(prefix="theta = t (rad) index: "),
                  steps=[dict(method="animate", label=f"{t[i]:.1f}", args=[[str(i)], dict(mode="immediate", frame=dict(duration=0, redraw=True))])
                         for i in range(0, len(t), 3)])])
fig.update_xaxes(range=[-1.3, 1.3], scaleanchor="y", row=1, col=1)
fig.update_yaxes(range=[-1.3, 1.3], row=1, col=1)
fig.update_xaxes(range=[0, stopt], row=1, col=2)
fig.update_yaxes(range=[-1.3, 1.3], row=1, col=2)
show(fig, "Phasor and sinusoid (phasor.m)", xlab=None, h=480)
print(f"frames = {len(t)}, theta from 0 to {t[-1]:.4f} rad ({t[-1] / np.pi:.3f} pi)")
print(f"check: |e^(j t)| = 1 for all frames: max | |x+jy| - 1 | = {np.max(np.abs(np.abs(x + 1j * y) - 1)):.1e}")

frames = 73, theta from 0 to 7.2000 rad (2.292 pi)
check: |e^(j t)| = 1 for all frames: max | |x+jy| - 1 | = 2.2e-16


## 2. 弦波的相位差（`phasedif.m`）

```matlab
dt=0.005; w=2; fi=pi/2; t=0:dt:3*pi;
y1=sin(w*t);  y2=sin(w*t+fi);
plot(t,y1,'red',t,y2,'blue')
```
$y_2=\sin(\omega t+\phi)$ 比 $y_1=\sin\omega t$ **超前** $\phi$；在時間軸上，$y_2$ 的每個峰值比 $y_1$ 早 $\Delta t=\phi/\omega$ 出現。

In [3]:
dt, w, fi = 0.005, 2, np.pi / 2
t = np.arange(0, 3 * np.pi + 1e-9, dt)
y1, y2 = np.sin(w * t), np.sin(w * t + fi)
fig = go.Figure([go.Scatter(x=t, y=y1, name="y1 = sin(2t)", line=dict(color="red")),
                 go.Scatter(x=t, y=y2, name="y2 = sin(2t + pi/2)", line=dict(color="blue"))])
show(fig, "Phase difference (phasedif.m)", ylab="y")
t1 = t[np.argmax(y1[: int(np.pi / w / dt)])]          # 第一個峰值
t2 = t[np.argmax(y2[: int(np.pi / w / dt)])]
print(f"first peak: y1 at t = {t1:.4f} s, y2 at t = {t2:.4f} s, lead = {t1 - t2:.4f} s; phi/w = {fi / w:.4f} s")

first peak: y1 at t = 0.7850 s, y2 at t = 0.0000 s, lead = 0.7850 s; phi/w = 0.7854 s


## 3. 同頻率弦波相加＝相量相加（網頁「弦波之和（相同頻率）」）

$A_1\sin(\omega t+\phi_1)+A_2\sin(\omega t+\phi_2)=A\sin(\omega t+\phi)$，其中 $Ae^{j\phi}=A_1e^{j\phi_1}+A_2e^{j\phi_2}$。

In [4]:
A1, p1, A2, p2, w = 1.0, 0.0, 0.8, 2 * np.pi / 3, 1.5
P = A1 * np.exp(1j * p1) + A2 * np.exp(1j * p2)
t = np.linspace(0, 10, 1001)
s = A1 * np.sin(w * t + p1) + A2 * np.sin(w * t + p2)
fig = go.Figure([go.Scatter(x=t, y=A1 * np.sin(w * t + p1), name="A1 sin(wt+p1)", line=dict(dash="dot")),
                 go.Scatter(x=t, y=A2 * np.sin(w * t + p2), name="A2 sin(wt+p2)", line=dict(dash="dot")),
                 go.Scatter(x=t, y=s, name="sum", line=dict(width=3))])
show(fig, "Sum of two sinusoids of the same frequency", ylab="y")
print(f"phasor sum: A = {abs(P):.4f}, phi = {np.degrees(np.angle(P)):.2f} deg; "
      f"max |sum - A sin(wt+phi)| = {np.max(np.abs(s - abs(P) * np.sin(w * t + np.angle(P)))):.1e}")

phasor sum: A = 0.9165, phi = 49.11 deg; max |sum - A sin(wt+phi)| = 1.4e-15


## 4. 拍：波形與聲音（`beatsound.m`）

```matlab
fs = 10000;  t = 0:1/fs:1.5;
x = sin(10*2*pi*t).*sin(440*2*pi*t);
sound(x,fs)
plot(t,x);  axis([0 0.2 -1 1])
```
440 Hz 的音被 10 Hz 的弦波調變。由積化和差
$\sin a\,\sin b=\tfrac12\left[\cos(a-b)-\cos(a+b)\right]$，它等於 **430 Hz 與 450 Hz 兩個音相減**：
兩個頻率相近的弦波相加（減），振幅包絡線以 $|\sin(2\pi\cdot10t)|$ 起伏，每秒「嗡」20 次（拍頻＝450−430＝20 Hz）。
執行後下方會出現播放器（對應 MATLAB 的 `sound`）。

In [5]:
fs = 10000
t = np.arange(0, 1.5 + 1e-12, 1 / fs)
x = np.sin(10 * 2 * np.pi * t) * np.sin(440 * 2 * np.pi * t)
x_sum = 0.5 * (np.cos(2 * np.pi * 430 * t) - np.cos(2 * np.pi * 450 * t))
m = t <= 0.2
fig = go.Figure([go.Scatter(x=t[m], y=x[m], name="x(t)", line=dict(width=1)),
                 go.Scatter(x=t[m], y=np.abs(np.sin(10 * 2 * np.pi * t[m])), name="envelope", line=dict(color="red", dash="dash")),
                 go.Scatter(x=t[m], y=-np.abs(np.sin(10 * 2 * np.pi * t[m])), showlegend=False, line=dict(color="red", dash="dash"))])
show(fig, "Beating: sin(2 pi 10 t) sin(2 pi 440 t) (beatsound.m)", xlab="Time (sec)", ylab="x(t)")
print(f"product = (cos 430 Hz - cos 450 Hz)/2: max difference = {np.max(np.abs(x - x_sum)):.1e}")
spec = np.abs(np.fft.rfft(x)) * 2 / len(x)
f = np.fft.rfftfreq(len(x), 1 / fs)
top = np.sort(f[np.argsort(spec)[-2:]])
print(f"spectrum peaks at {top[0]:.1f} Hz and {top[1]:.1f} Hz, beat frequency = {top[1] - top[0]:.1f} Hz")
display(Audio(x, rate=fs))

product = (cos 430 Hz - cos 450 Hz)/2: max difference = 7.2e-13
spectrum peaks at 430.0 Hz and 450.0 Hz, beat frequency = 20.0 Hz
